# IW3 2D-to-3D Stereoscopic Video & Image Converter

This notebook can convert 2D videos and photos into immersive 3D stereoscopic media (Full Side-by-Side, Half SBS, Top-Bottom, VR180, Anaglyph) using [`nagadomi/nunif`](https://github.com/nagadomi/nunif)'s **`iw3`** module.

---
Ensure your Colab runtime is set to GPU (`Runtime` → `Change runtime type` → select `T4 GPU` or higher).


In [ ]:
# @title 1. Verify GPU Runtime { display-mode: "form" }
# @markdown Inspect hardware allocation and verify CUDA device availability.

import shutil
import sys
import torch

print("Checking compute environment...")
if not torch.cuda.is_available():
    raise RuntimeError(
        "CUDA GPU not detected! Please go to Runtime -> Change runtime type, "
        "select a GPU accelerator (e.g., T4 GPU), and restart the session."
    )

gpu_name = torch.cuda.get_device_name(0)
gpu_props = torch.cuda.get_device_properties(0)
total_vram_gb = gpu_props.total_memory / (1024 ** 3)

print(f"✓ Active GPU: {gpu_name}")
print(f"✓ VRAM Total: {total_vram_gb:.2f} GB")
print(f"✓ CUDA Version: {torch.version.cuda}")
print(f"✓ PyTorch Version: {torch.__version__}")

ffmpeg_path = shutil.which("ffmpeg")
ffprobe_path = shutil.which("ffprobe")
if ffmpeg_path and ffprobe_path:
    print(f"✓ FFmpeg/FFprobe binaries detected at: {ffmpeg_path}")
else:
    print("⚠ FFmpeg or FFprobe binary missing in PATH. Video processing requires both tools.")


In [ ]:
# @title 2. Mount Google Drive { display-mode: "form" }
# @markdown Mount Google Drive to read inputs and write outputs without risking timeout data loss.

import os

mount_drive = True # @param {type:"boolean"}
drive_mount_point = "/content/drive" # @param {type:"string"}
default_workspace = "/content/drive/MyDrive/iw3" # @param {type:"string"}

if mount_drive:
    try:
        from google.colab import drive
        if not os.path.ismount(drive_mount_point):
            print(f"Mounting Google Drive to {drive_mount_point}...")
            drive.mount(drive_mount_point)
            print("✓ Google Drive mounted successfully.")
        else:
            print("✓ Google Drive already mounted.")
        
        os.makedirs(os.path.join(default_workspace, "inputs"), exist_ok=True)
        os.makedirs(os.path.join(default_workspace, "outputs"), exist_ok=True)
        print(f"✓ Workspace directory ready: {default_workspace}")
    except ImportError:
        print("⚠ google.colab module not found (running in local container). Skipping Drive mount.")
        os.makedirs("./workspace/inputs", exist_ok=True)
        os.makedirs("./workspace/outputs", exist_ok=True)
else:
    print("Drive mounting skipped by user parameter. Using local storage.")


In [ ]:
# @title 3. Setup IW3 Environment { display-mode: "form" }
# @markdown Clone nunif repository, install dependencies, and download depth models.

import os
import subprocess
import sys

force_reinstall = False # @param {type:"boolean"}
nunif_dir = "/content/nunif"

# 1. Clone repository
if not os.path.exists(nunif_dir) or force_reinstall:
    if os.path.exists(nunif_dir):
        print("Removing existing nunif directory...")
        subprocess.run(["rm", "-rf", nunif_dir], check=True)
    print("Cloning nagadomi/nunif repository...")
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/nagadomi/nunif.git", nunif_dir], check=True)
    print("✓ nagadomi/nunif cloned successfully.")
else:
    print(f"✓ Existing nunif repository detected at {nunif_dir}.")

# 2. Add nunif to Python sys.path
if nunif_dir not in sys.path:
    sys.path.insert(0, nunif_dir)

# 3. Install nunif requirements quietly
print("Installing dependencies (requirements.txt, PyAV, onnxruntime-gpu, safetensors, einops)...")
req_file = os.path.join(nunif_dir, "requirements.txt")
install_cmd = [
    sys.executable, "-m", "pip", "install", "-q",
    "-r", req_file,
    "av", "onnxruntime-gpu", "safetensors", "einops"
]
subprocess.run(install_cmd, check=True)
print("✓ Environment dependencies ready.")

# 4. Pre-download Depth Anything V2 Large checkpoint (~1.3 GB)
ckpt_dir = os.path.join(nunif_dir, "iw3", "pretrained_models", "hub", "checkpoints")
os.makedirs(ckpt_dir, exist_ok=True)
vitl_ckpt = os.path.join(ckpt_dir, "depth_anything_v2_vitl.pth")
if not os.path.exists(vitl_ckpt):
    print("Downloading Depth Anything V2 Large checkpoint (~1.3 GB)...")
    hf_url = "https://huggingface.co/depth-anything/Depth-Anything-V2-Large/resolve/main/depth_anything_v2_vitl.pth"
    subprocess.run(["wget", "-q", "--show-progress", "-O", vitl_ckpt, hf_url], check=True)
    print("✓ Depth Anything V2 Large checkpoint ready.")
else:
    print("✓ Depth Anything V2 Large checkpoint already present.")

# 5. Verify module availability
try:
    import iw3
    import av
    print(f"✓ iw3 and PyAV (v{av.__version__}) successfully accessible.")
except ImportError as err:
    raise ImportError(f"Failed to import required modules: {err}")


In [ ]:
# @title 4. Run 2D to 3D Conversion { display-mode: "form" }
# @markdown Configure source media, stereoscopic layout, inference parameters, and encoding options.

import os
import shutil
import subprocess
import sys

# --- Storage & Naming ---
# @markdown ### 1. Storage & Naming
# @markdown - `input_path`: Path to 2D image, video file, or folder of media.
# @markdown - `output_path`: Target destination file or folder.
# @markdown - `auto_vr_naming`: Automatically appends standard VR tags (e.g., `_LRF_Full_SBS`, `_180x180_LR`) so VR players (SKYBOX, 4XVR, Pigasus) autodetect 3D format without manual setup.
# @markdown - `resume`: Skips processing if target output file already exists. Useful to resume interrupted batch processing or recover after Colab disconnects without re-rendering.
input_path = "/content/drive/MyDrive/iw3/input.mp4" # @param {type:"string"}
output_path = "/content/drive/MyDrive/iw3/output" # @param {type:"string"}
auto_vr_naming = True # @param {type:"boolean"}
resume = True # @param {type:"boolean"}

# --- Stereoscopic Geometry ---
# @markdown ### 2. Stereoscopic Geometry
# @markdown - `stereo_format`: `full-sbs` (double-width, for VR headsets), `half-sbs` (half per eye for legacy 3D TVs and headsets), `tb`/`half-tb` (Top-Bottom), `vr180` (hemispherical VR), `anaglyph` (Red/Cyan glasses), or `cross-eyed`.
# @markdown - `divergence`: 3D effect depth intensity. Higher values increase separation; excessive values cause eye strain.
# @markdown - `convergence`: Zero-parallax plane (screen focal surface). `1.0` is recommended for curved screens.
# @markdown - `synthetic_view`: `both` balances warping evenly between eyes. `left` or `right` keeps one eye with zero distortion and warps only the opposite eye.
# @markdown - `ipd_offset`: IPD offset percentage. Adjusts perceived world scale to fix scenes feeling miniature or giant.
# @markdown - `preserve_screen_border`: Forces zero parallax at screen borders to prevent visual discomfort caused when 3D objects clip against the left/right frame edges in VR.
stereo_format = "full-sbs" # @param ["full-sbs", "half-sbs", "tb", "half-tb", "vr180", "anaglyph", "cross-eyed", "rgbd", "half-rgbd"]
anaglyph_type = "dubois" # @param ["dubois", "dubois2", "color", "gray", "half-color", "wimmer", "wimmer2"]
divergence = 2.0 # @param {type:"slider", min:0.5, max:5.0, step:0.1}
convergence = 0.5 # @param {type:"slider", min:0.0, max:1.0, step:0.05}
synthetic_view = "both" # @param ["both", "right", "left"]
ipd_offset = 0.0 # @param {type:"slider", min:0.0, max:10.0, step:0.5}
preserve_screen_border = False # @param {type:"boolean"}

# --- Model & Warping Engine ---
# @markdown ### 3. Depth Model & Warping Engine
# @markdown - `depth_model`: For static photos/art, `Any_V2_L` yields superior spatial details. For videos, `VDA_L` (Video Depth Anything) ensures continuous temporal coherence without flickering.
# @markdown - `method`: Warping algorithm. `row_flow_v3` is the default trained model.
# @markdown - `resolution`: Depth inference height. Higher value captures more details but takes more time and needs more VRAM.
# @markdown - `batch_size`: Batch size for GPU inference.
depth_model = "Any_V2_L" # @param ["Any_V2_L", "Any_V2_B", "Any_V2_S", "VDA_L", "VDA_B", "Any_V3_Mono", "ZoeD_Any_N"]
method = "row_flow_v3" # @param ["row_flow_v3", "row_flow_v3_sym", "row_flow_v2", "forward_fill", "mlbw_l2", "mlbw_l4"]
resolution = 518 # @param [320, 384, 392, 504, 518] {type:"raw"}
batch_size = 2 # @param {type:"integer"}

# --- Artifact Mitigation & Video Stability ---
# @markdown ### 4. Artifact Suppression & Video Stability
# @markdown - `edge_dilation`: Dilates foreground masks to eliminate edge tearing and halos around subjects.
# @markdown - `ema_normalize`: Exponential Moving Average depth normalization to prevent video depth flickering. Highly recommended for video.
# @markdown - `scene_detect`: Uses TransNetV2 to detect shot transitions and reset EMA depth history across cuts, avoiding depth smearing. Highly recommended for video.
# @markdown - `foreground_scale`: Scales depth of foreground subjects. Use positive values for outdoor photos where people look flat like cardboard. Keep `0.0` for video.
edge_dilation = 2 # @param {type:"slider", min:0, max:4, step:1}
ema_normalize = True # @param {type:"boolean"}
scene_detect = True # @param {type:"boolean"}
foreground_scale = 0.0 # @param {type:"slider", min:-3.0, max:3.0, step:0.5}

# --- Video Encoding & Frame Trimming ---
# @markdown ### 5. Video Encoding & Frame Trimming
# @markdown - `framerate_mode`: `original` preserves full fps source framerate without downsampling. `30` or `60` sets an explicit cap.
# @markdown - `crf`: Constant Rate Factor quality (lower is higher quality). `17`-`19` recommended for high-clarity VR.
# @markdown - `autocrop`: Detects and crops black letterbox borders so they don't get warped into 3D space.
# @markdown - `start_time` / `end_time`: Optional timestamp offsets (`00:01:00`) to test a short clip.
framerate_mode = "original" # @param ["original", "30", "60"]
video_codec = "h264_nvenc" # @param ["h264_nvenc", "hevc_nvenc", "libx264", "libx265"]
crf = 20 # @param {type:"slider", min:14, max:28, step:1}
preset = "medium" # @param ["medium", "fast", "slow", "p4", "p5", "p6"]
pix_fmt = "yuv420p" # @param ["yuv420p", "yuv444p", "yuv420p10le"]
colorspace = "auto" # @param ["auto", "bt709", "bt601", "bt2020-tv"]
autocrop = "NONE" # @param ["NONE", "BLACK_TB", "BLACK", "FLAT_TB", "FLAT"]
hwaccel = "none" # @param ["none", "cuda", "cuda_hwdownload"]
start_time = "" # @param {type:"string"}
end_time = "" # @param {type:"string"}

# --- Execution Pipeline ---
nunif_dir = "/content/nunif"
if not os.path.exists(nunif_dir):
    raise FileNotFoundError(
        f"nunif directory not found at {nunif_dir}. Please execute Cell 3 first."
    )

if not os.path.exists(input_path):
    raise FileNotFoundError(
        f"Input path does not exist: {input_path}. Please verify the file or directory path."
    )

# Automatic Checkpoint Download
ckpt_dir = os.path.join(nunif_dir, "iw3", "pretrained_models", "hub", "checkpoints")
os.makedirs(ckpt_dir, exist_ok=True)
ckpt_map = {
    "Any_V2_L": ("depth_anything_v2_vitl.pth", "https://huggingface.co/depth-anything/Depth-Anything-V2-Large/resolve/main/depth_anything_v2_vitl.pth"),
    "Any_V2_B": ("depth_anything_v2_vitb.pth", "https://huggingface.co/depth-anything/Depth-Anything-V2-Base/resolve/main/depth_anything_v2_vitb.pth"),
    "Any_V2_S": ("depth_anything_v2_vits.pth", "https://huggingface.co/depth-anything/Depth-Anything-V2-Small/resolve/main/depth_anything_v2_vits.pth"),
    "VDA_L": ("video_depth_anything_vitl.pth", "https://huggingface.co/depth-anything/Video-Depth-Anything-Large/resolve/main/video_depth_anything_vitl.pth"),
    "VDA_B": ("video_depth_anything_vitb.pth", "https://huggingface.co/depth-anything/Video-Depth-Anything-Base/resolve/main/video_depth_anything_vitb.pth"),
    "Any_V3_Mono": ("da3mono-large.safetensors", "https://huggingface.co/ByteDance-Seed/Depth-Anything-3/resolve/main/da3mono-large.safetensors"),
}
if depth_model in ckpt_map:
    filename, url = ckpt_map[depth_model]
    target_file = os.path.join(ckpt_dir, filename)
    if not os.path.exists(target_file):
        print(f"Downloading {depth_model} checkpoint ({filename})...")
        subprocess.run(["wget", "-q", "--show-progress", "-O", target_file, url], check=True)
        print(f"✓ Checkpoint ready: {filename}")

# Compute Target Filename / Directory with Standard VR Player Suffix
vr_suffix_map = {
    "full-sbs": "_LRF_Full_SBS",
    "half-sbs": "_LR",
    "tb": "_TBF_fulltb",
    "half-tb": "_TB",
    "vr180": "_180x180_LR",
    "anaglyph": f"_redcyan_{anaglyph_type}",
    "cross-eyed": "_RLF_cross",
    "rgbd": "_RGBD",
    "half-rgbd": "_HRGBD",
}
vr_tag = vr_suffix_map.get(stereo_format, "_LRF_Full_SBS")

resolved_output = output_path.strip()
is_input_dir = os.path.isdir(input_path)

if is_input_dir:
    # Batch Directory Mode: output MUST remain a directory. iw3 names each output file automatically.
    os.makedirs(resolved_output, exist_ok=True)
else:
    # Single File Mode:
    if os.path.isdir(resolved_output) or not os.path.splitext(resolved_output)[1]:
        os.makedirs(resolved_output, exist_ok=True)
        input_base = os.path.splitext(os.path.basename(input_path))[0]
        is_video = input_path.lower().endswith((".mp4", ".mkv", ".avi", ".mov", ".webm", ".m4v"))
        ext = ".mp4" if is_video else ".png"
        resolved_output = os.path.join(resolved_output, f"{input_base}{vr_tag}{ext}")
    elif auto_vr_naming:
        parent_dir = os.path.dirname(os.path.abspath(resolved_output))
        base_name, ext = os.path.splitext(os.path.basename(resolved_output))
        if not any(tag in base_name for tag in ["_LRF", "_Full_SBS", "_LR", "_TB", "_180x180", "_redcyan", "_RLF"]):
            resolved_output = os.path.join(parent_dir, f"{base_name}{vr_tag}{ext}")

out_parent = os.path.dirname(os.path.abspath(resolved_output))
if out_parent:
    os.makedirs(out_parent, exist_ok=True)

# Build CLI Command
cmd = [
    sys.executable, "-m", "iw3",
    "--input", input_path,
    "--output", resolved_output,
    "--depth-model", str(depth_model),
    "--divergence", str(float(divergence)),
    "--convergence", str(float(convergence)),
    "--method", str(method),
    "--resolution", str(int(resolution)),
    "--batch-size", str(int(batch_size)),
    "--edge-dilation", str(int(edge_dilation)),
    "--synthetic-view", str(synthetic_view),
    "--video-codec", str(video_codec),
    "--crf", str(int(crf)),
    "--preset", str(preset),
    "--pix-fmt", str(pix_fmt),
    "--colorspace", str(colorspace),
    "--yes"
]

# Stereo Format Flag Mapping
if stereo_format == "half-sbs":
    cmd.append("--half-sbs")
elif stereo_format == "tb":
    cmd.append("--tb")
elif stereo_format == "half-tb":
    cmd.append("--half-tb")
elif stereo_format == "vr180":
    cmd.append("--vr180")
elif stereo_format == "anaglyph":
    cmd.extend(["--anaglyph", str(anaglyph_type)])
elif stereo_format == "cross-eyed":
    cmd.append("--cross-eyed")
elif stereo_format == "rgbd":
    cmd.append("--rgbd")
elif stereo_format == "half-rgbd":
    cmd.append("--half-rgbd")

# Optional Parameters
if float(foreground_scale) != 0.0:
    cmd.extend(["--foreground-scale", str(float(foreground_scale))])
if float(ipd_offset) > 0.0:
    cmd.extend(["--ipd-offset", str(float(ipd_offset))])
if preserve_screen_border:
    cmd.append("--preserve-screen-border")
if ema_normalize:
    cmd.append("--ema-normalize")
if scene_detect:
    cmd.append("--scene-detect")
if resume:
    cmd.append("--resume")

if framerate_mode == "original":
    cmd.extend(["--max-fps", "1000"])
else:
    cmd.extend(["--max-fps", str(int(framerate_mode))])

if autocrop != "NONE":
    cmd.extend(["--autocrop", str(autocrop)])
if hwaccel != "none":
    cmd.extend(["--hwaccel", str(hwaccel)])
if start_time.strip():
    cmd.extend(["--start-time", start_time.strip()])
if end_time.strip():
    cmd.extend(["--end-time", end_time.strip()])

# Export resolved target for verification cell
os.environ["IW3_LAST_OUTPUT"] = resolved_output

print("=" * 60)
print("Starting IW3 2D-to-3D Conversion Pipeline")
print(f"Input:             {input_path}")
print(f"Resolved Output:   {resolved_output}")
print(f"Model:             {depth_model} (Resolution: {resolution})")
print(f"Warping Method:    {method}")
print(f"Stereo Geometry:   {stereo_format} (Divergence: {divergence}, Convergence: {convergence})")
print(f"Stability:         EMA Normalize={ema_normalize}, Scene Detect={scene_detect}, Edge Fix={edge_dilation}")
print(f"Video Encoding:    {video_codec} (CRF: {crf}, Preset: {preset}, PixFmt: {pix_fmt})")
print("=" * 60)
print(f"Executing CLI: {' '.join(cmd)}\n")

# Stream Subprocess Execution in Real-time
process = subprocess.Popen(
    cmd,
    cwd=nunif_dir,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1
)

output_lines = []
try:
    for line in iter(process.stdout.readline, ''):
        print(line, end='', flush=True)
        output_lines.append(line)
except KeyboardInterrupt:
    process.terminate()
    print("\nProcess interrupted by user.")
    raise

process.wait()

if process.returncode != 0:
    full_log = "".join(output_lines)
    if "CUDA out of memory" in full_log:
        print("\n" + "!" * 60)
        print("ERROR: CUDA Out of Memory (OOM) encountered!")
        print("Remediation steps:")
        print("  1. Reduce 'resolution' to 384 or 392 in the form above.")
        print("  2. Switch 'depth_model' to 'Any_V2_B' or 'Any_V2_S'.")
        print("  3. Set 'batch_size' to 1.")
        print("!" * 60)
    raise RuntimeError(f"iw3 conversion failed with exit code {process.returncode}")

print("\n" + "=" * 60)
print("✓ Conversion completed successfully!")
print(f"✓ Output saved to: {resolved_output}")
print("=" * 60)


In [ ]:
# @title 5. Verify Output Stream { display-mode: "form" }
# @markdown Inspect container integrity, assert stereoscopic geometry, and render a visual sample.

import os
import shutil
import subprocess
import sys

# Retrieve last output path if set
default_target = os.environ.get("IW3_LAST_OUTPUT", "/content/drive/MyDrive/iw3/output")
target_video_path = default_target # @param {type:"string"}
expected_format = "full-sbs" # @param ["full-sbs", "half-sbs", "tb", "half-tb", "vr180", "anaglyph", "cross-eyed"]
run_null_mux_check = True # @param {type:"boolean"}
extract_preview_frame = True # @param {type:"boolean"}
preview_timestamp = "00:00:02" # @param {type:"string"}

# If directory was specified, pick the newest generated media file inside
if os.path.isdir(target_video_path):
    candidate_files = [
        os.path.join(target_video_path, f)
        for f in os.listdir(target_video_path)
        if f.lower().endswith((".mp4", ".mkv", ".avi", ".png", ".jpg", ".webp"))
    ]
    if candidate_files:
        candidate_files.sort(key=os.path.getmtime, reverse=True)
        target_video_path = candidate_files[0]
        print(f"Detected latest media in directory: {target_video_path}")

if not os.path.exists(target_video_path):
    raise FileNotFoundError(
        f"Target file does not exist: {target_video_path}. Please check the path or run Cell 4."
    )

file_size_bytes = os.path.getsize(target_video_path)
if file_size_bytes == 0:
    raise ValueError(f"Target file is empty (0 bytes): {target_video_path}")

file_size_mb = file_size_bytes / (1024 * 1024)
print("=" * 60)
print("Stereoscopic Media Verification")
print(f"File: {target_video_path}")
print(f"Size: {file_size_mb:.2f} MB")
print("=" * 60)

ffprobe_bin = shutil.which("ffprobe")
if not ffprobe_bin:
    raise RuntimeError("ffprobe binary not found in PATH.")

# 1. Stream Metadata Inspection
probe_cmd = [
    ffprobe_bin, "-v", "error",
    "-select_streams", "v:0",
    "-show_entries", "stream=width,height,r_frame_rate,codec_name,pix_fmt,duration",
    "-of", "default=noprint_wrappers=1",
    target_video_path
]
probe_result = subprocess.run(probe_cmd, capture_output=True, text=True)
if probe_result.returncode != 0:
    raise RuntimeError(f"ffprobe failed to inspect media: {probe_result.stderr}")

probe_info = dict(
    line.split("=", 1) for line in probe_result.stdout.strip().splitlines() if "=" in line
)

width = int(probe_info.get("width", 0))
height = int(probe_info.get("height", 0))
codec = probe_info.get("codec_name", "unknown")
pix_fmt = probe_info.get("pix_fmt", "unknown")
duration = probe_info.get("duration", "N/A")
fps = probe_info.get("r_frame_rate", "unknown")

print("\n--- Stream Geometry & Encoding ---")
print(f"Resolution:   {width}x{height}")
print(f"Codec:        {codec}")
print(f"Pixel Format: {pix_fmt}")
print(f"Frame Rate:   {fps}")
print(f"Duration:     {duration}s")

# 2. Geometry Assertion
print("\n--- Stereoscopic Geometry Check ---")
aspect_ratio = width / height if height > 0 else 0
if expected_format == "full-sbs":
    print(f"Reported Aspect Ratio: {aspect_ratio:.2f}:1")
    if aspect_ratio >= 2.5:
        print("✓ Geometry conforms to Full Side-by-Side (~32:9 for 16:9 source).")
    else:
        print(f"⚠ Detected aspect ratio is {aspect_ratio:.2f}:1 (Full SBS typically >= 2.5:1).")
elif expected_format == "half-sbs":
    print(f"Reported Aspect Ratio: {aspect_ratio:.2f}:1")
    print("✓ Geometry conforms to Half Side-by-Side.")
elif "tb" in expected_format:
    print(f"Reported Aspect Ratio: {aspect_ratio:.2f}:1")
    print("✓ Geometry conforms to Top-Bottom layout.")
else:
    print(f"✓ Geometry recorded for format '{expected_format}'.")

# 3. Null-Mux Stream Integrity Check
is_video = target_video_path.lower().endswith((".mp4", ".mkv", ".avi", ".mov", ".webm", ".m4v"))
if run_null_mux_check and is_video:
    print("\n--- Null-Mux Stream Integrity Check ---")
    ffmpeg_bin = shutil.which("ffmpeg")
    null_mux_cmd = [
        ffmpeg_bin, "-v", "error",
        "-i", target_video_path,
        "-f", "null", "-"
    ]
    null_res = subprocess.run(null_mux_cmd, capture_output=True, text=True)
    if null_res.returncode == 0 and not null_res.stderr.strip():
        print("✓ Stream integrity verified: 0 packet decoding errors, valid timestamps.")
    else:
        print("⚠ FFmpeg reported warnings during stream decode:")
        print(null_res.stderr.strip())

# 4. Inline Visual Stereoscopic Preview
if extract_preview_frame:
    print("\n--- Visual Stereoscopic Preview ---")
    ffmpeg_bin = shutil.which("ffmpeg")
    preview_output = "/tmp/iw3_preview_frame.png"
    if is_video:
        frame_cmd = [
            ffmpeg_bin, "-y", "-v", "error",
            "-ss", preview_timestamp,
            "-i", target_video_path,
            "-vframes", "1",
            preview_output
        ]
        subprocess.run(frame_cmd, capture_output=True, text=True)
    else:
        preview_output = target_video_path
    
    if os.path.exists(preview_output):
        try:
            from IPython.display import Image, display
            print(f"✓ Displaying stereoscopic preview:")
            display(Image(preview_output, width=720))
        except ImportError:
            print(f"✓ Preview saved to {preview_output} (IPython display unavailable).")

print("\n" + "=" * 60)
print("✓ Verification process complete.")
print("=" * 60)
